
# Getting Started with Kaggriculture 🚜

Welcome to Kaggriculture! 

**What is this notebook?**
This notebook is your beginner-friendly guide to understanding and playing the Kaggriculture farming simulation on Kaggle. We will walk through the core mechanics, build a minimal starter agent step-by-step, and show you how to test your agent before submitting it to the competition. 

**What you will learn:**
- The basic rules of the game (using a real-world farming analogy).
- How to set up the environment and inspect the game "dashboard".
- How to write a simple Python agent that plants, waters, and sells wheat.
- How to run a local simulation and read the results.
- Exactly how to submit your final agent to the Kaggle leaderboard.

Let's get farming!



## 1. Environment Setup 🛠️

First, we need to install the game environment and create our virtual farm. We'll set `debug=True` so we can see what's happening if anything goes wrong.


In [ ]:

# Install the latest Kaggle environments package
# !pip install -U kaggle-environments

from kaggle_environments import make

# Create the environment!
env = make("kaggriculture", debug=True)
print("Environment created successfully!")



## 2. Understanding the Game 🌻

Think of Kaggriculture exactly like running a small farm and a stall at a bustling farmers market. 
You and your opponent each start with a small plot of land and a bit of cash. You have to decide which seeds to buy, when to plant them, and when to harvest. You can even hire farm hands to help you do more chores at once!

### The Core Loop: Days, Turns, and Deadlines ⏰
A full "season" is 30 days long, and each day is broken into 24 turns (or "hours"). So, a single game is exactly 720 turns. 

Just like real life, plants need care. You must water your crops every single day. If you neglect a plant and let it go unwatered for two days in a row, it's just like leaving your houseplants to fend for themselves while you go on a weekend vacation — they will wither and turn into weeds!

### The Market: Supply and Demand 📉📈
When you harvest your crops, you'll sell them at the town market. The market has a dynamic economy. If you and the other farmer both show up with hundreds of tomatoes, the townspeople aren't going to pay top dollar for them. The price will drop! Conversely, if nobody has sold wheat in a while, the town will be hungry for it, and the price will go up.

### The Dashboard (Observation) 📊
Every turn, the game hands you a "dashboard" (the observation). This dictionary tells you everything you need to know:
- `obs["player"]`: Which player you are (0 or 1).
- `obs["step"]`: What turn it is (0 to 719).
- `obs["farms"]`: A summary of both farms, including bank balances and grid maps.
- `obs["market"]`: The current prices for all goods.

Let's peek at the very first dashboard of the game:


In [ ]:

# Get the initial observation from the environment
initial_state = env.reset()
obs = initial_state[0].observation

player_index = obs["player"]
my_farm = obs["farms"][player_index]

print(f"I am player: {player_index}")
print(f"My starting money: ${my_farm['money']}")
print(f"Initial WHEAT price: ${obs['market']['prices']['WHEAT']}")



## 3. Agent Implementation 🧑‍🌾

To play the game, we need to write an "agent" — a Python function that looks at the dashboard and decides what to do. 
Let's build a simple Wheat farmer step-by-step. First, we need to buy seeds and sell our harvest. We'll write a small helper function for this.


In [ ]:

def handle_market(private_info, my_farm):
    orders = []
    # If we have no wheat seeds and enough money, buy 1 seed
    if private_info["seeds"].get("WHEAT", 0) == 0 and my_farm["money"] >= 10:
        orders.append(["BUY_SEED", "WHEAT", 1])
    
    # If we have harvested wheat in our shed, sell it!
    wheat_in_shed = private_info["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        orders.append(["SELL", "WHEAT", wheat_in_shed])
        
    return orders



Now, what should the farmer physically do on the farm? If they are standing on an empty plot of dirt and have a seed in their pocket, they should plant it!


In [ ]:

def handle_planting(tile, private_info):
    # If the tile is empty and we have a seed, plant it!
    if tile is None and private_info["seeds"].get("WHEAT", 0) > 0:
        return ["PLANT", "WHEAT"]
    return None



But what if there is already a plant growing? We need to make sure we water it so it doesn't dry up, and harvest it when it's fully grown.


In [ ]:

def handle_plant_care(tile, current_day):
    # Check if the tile has a plant
    if isinstance(tile, dict) and tile.get("kind") == "PLANT":
        crop_age = current_day - tile["planted_day"]
        
        if crop_age >= 2:  # Wheat is fully grown after 2 days
            return ["HARVEST"]
        if not tile["watered_today"]:
            return ["WATER"]
            
    return None



Finally, let's tie it all together into our main agent function. The game will call this function every single turn!


In [ ]:

def my_wheat_farmer(obs):
    p = obs["player"]
    my_farm = obs["farms"][p]
    priv = obs["private"]
    
    fx, fy = my_farm["farmer"]
    tile = my_farm["tiles"][fy][fx]
    
    orders = handle_market(priv, my_farm)
    action = handle_planting(tile, priv)
    
    if action is None:
        action = handle_plant_care(tile, obs["day"])
    if action is None:
        action = ["PASS"]
        
    return {"farmer": action, "hands": [], "market": orders}



## 4. Running the Simulation 🎮

Our agent is ready! Let's pit our newly minted wheat farmer against a completely random opponent for a full season.


In [ ]:

print("Running a full season...")
# We run our farmer as Player 1, and the built-in "random" agent as Player 2
my_player_index = 0
env.run([my_wheat_farmer, "random"])

final_rewards = [s.reward for s in env.steps[-1]]
print(f"Game finished! Our reward (money): ${final_rewards[my_player_index]}")
print(f"Opponent reward (money): ${final_rewards[1 - my_player_index]}")



## 5. Observing Results 🔍

To keep this notebook lightweight, instead of generating a massive interactive visualizer, we will print a text-based summary of our farm's progress throughout the season. Let's look at our money and wheat stock every 120 turns (5 days).


In [ ]:

print(f"{'Turn':<6} | {'Day':<4} | {'My Money':<10} | {'Wheat in Shed':<15}")
print("-" * 45)

for step_idx in range(0, len(env.steps), 120):
    # Dynamically reference our player's state based on our assigned index
    my_step_data = env.steps[step_idx][my_player_index]
    
    if my_step_data.observation and "farms" in my_step_data.observation:
        obs_at_step = my_step_data.observation
        p_idx = obs_at_step["player"]
        
        my_money = obs_at_step["farms"][p_idx]["money"]
        day = obs_at_step["day"]
        
        try:
            my_wheat = obs_at_step["private"]["shed"].get("WHEAT", 0)
            wheat_str = str(my_wheat)
        except KeyError as e:
            wheat_str = f"(Error: {e} not found)"
            
        print(f"{step_idx:<6} | {day:<4} | ${my_money:<9.1f} | {wheat_str:<15}")



## 6. Submission Instructions 🚀

Ready to climb the leaderboard? Here is exactly how to submit your agent to the Kaggriculture competition:

1. **Accept the Rules**: Before anything else, visit the [Kaggriculture competition page](https://www.kaggle.com/competitions/kaggriculture) and click **"Join Competition"** to accept the rules.
2. **Prepare Your File**: Your agent must be contained within a single `main.py` file (or a `tar.gz` archive with `main.py` at the root). 
3. **The Agent Function**: According to the official docs, your `main.py` file must contain a top-level function that accepts an `obs` dictionary and returns an action dictionary, matching the pattern shown in this notebook. The standard function name is `agent`.
4. **Local Testing**: Always test your `main.py` locally first by running `env.run(["main.py", "random"])` to ensure it doesn't crash.
5. **Submit via Kaggle**: 
   - **Through the UI**: Go to the competition's "Submit" tab and upload your `main.py` or `tar.gz` file.
   - **Through the CLI**: If you have the Kaggle CLI installed, run: `kaggle competitions submit kaggriculture -f main.py -m "My first submission!"`

*Note: Your agent runs in a strict sandboxed environment on Kaggle, so ensure you only rely on standard built-in libraries and the game's provided dependencies.*



## 7. Key Takeaways & Next Steps 🤔

You've built your first farm! But this is just the beginning. As you think about how to improve your agent, consider these questions:
- What do you think happens to the market price if you hoard 50 wheat and sell them all on the exact same turn?
- If you hire farm hands, the cost goes up in a Fibonacci sequence (1, 1, 2, 3, 5, 8...). Is there a point where hiring another hand isn't worth it? Like adding too many cooks to a tiny kitchen?
- Could you earn more money by planting Strawberries or Melons instead of Wheat? How would your timing change?
- Our farmer just stood in one spot. What if they walked around to plant a whole field? 

Happy farming! 🌾
